# Deploy Lakehouse

In [3]:
from pathlib import Path
from trino_stack.render import render_collective
from trino_stack.lakehouse import Lakehouse
import trino_stack

"""
Deploy lakehouse with the tpcds schema and register to HMS
"""

TRINO_STACK_DIR = Path(trino_stack.__file__).resolve().parent

res = render_collective(
    templates_dir=str(TRINO_STACK_DIR / "manifests"),
    values_path=str(TRINO_STACK_DIR / "lakehouses" / "lakehouse-tpcds-small.yaml"),
)

lh = Lakehouse(res, verbose=False)

lh.deploy()
lh.register_schema("tpcds", "/mnt/iceberg/warehouse", use_dot_db=True)  # <schema name in /warehouse>  <location of /warehouse in control container>


Deploying Trino
Checking Trino health ...
STATUS:

Lakehouse: lakehouse-g
Namespace: pgr24james
Selector:  app.kubernetes.io/instance=lakehouse-g
URL:       http://trino-route-lakehouse-g-pgr24james.apps.os.dcs.gla.ac.uk

Pods:

  Coordinator:
    - trino-coord-pod-lakehouse-g              Running    Ready     idagpu-12          10.130.16.207

  Metastore:
    - hive-metastore-postgres-lakehouse-g      Running    Ready     idagpu-05          10.131.4.205

  Worker:
    - trino-worker-lakehouse-g-0-0             Running    Ready     idagpu-22          10.130.13.76

Schemas:
  <none>
Waiting for Trino engine to ready.
Trino is ready.
Trino host: trino-service-lakehouse-g.pgr24james.svc.cluster.local
Schema:     tpcds
Warehouse:  /mnt/iceberg/warehouse
Tables:     24
Registered all tables


# Config

In [4]:
import json
from trino_stack.config import MODEL_NAME, BASE_MODEL_URL, API_KEY_ENV
from workload_generation.common import context_from_lakehouse

"""
Hardcoded config used by every generation cell below.
Edit these values to change schema / instance / model / volume.
"""

INSTANCE_NAME = "lakehouse-a"
NAMESPACE = "pgr24james"

SCHEMA = "tpcds"
CATALOG = "iceberg"

NUM_QUERIES = 100
MODEL_NAME_OVERRIDE = MODEL_NAME
BASE_URL = BASE_MODEL_URL
API_KEY_ENV_NAME = API_KEY_ENV
TEMPERATURE = 0.6
REASONING = "high"

RANDOM_SEED = 42
WARMUP = True

ctx = context_from_lakehouse(
    lh,
    schema=SCHEMA,
    catalog=CATALOG,
    model_name=MODEL_NAME_OVERRIDE,
    base_url=BASE_URL,
    api_key_env=API_KEY_ENV_NAME,
    temperature=TEMPERATURE,
    reasoning=REASONING,
)


# Generate Baseline Workloads

## sqlstorm

In [5]:
from workload_generation.baselines import sqlstorm

"""
Generate a workload using the sqlstorm baseline
"""

WORKLOAD_NAME_SQLSTORM = "sqlstorm_{}".format(SCHEMA)

report_sqlstorm = sqlstorm.generate_workload(
    ctx,
    workload_name=WORKLOAD_NAME_SQLSTORM,
    num_queries=NUM_QUERIES,
    warmup=WARMUP,
    random_seed=RANDOM_SEED,
)

print(json.dumps({
    "baseline": report_sqlstorm["baseline"],
    "workload_dir": report_sqlstorm["workload_dir"],
    "num_queries": report_sqlstorm["num_queries"],
    "duration_s": round(report_sqlstorm["duration_s"], 1),
}, indent=2, default=str))


2026-07-09 12:28:50,936 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"


Model warm-up response: ready


2026-07-09 12:29:06,597 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 12:29:22,438 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 12:29:29,916 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 12:30:06,889 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 12:30:09,831 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 12:30:14,238 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 12:31:00,998 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 12:31:16,377 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 12:31:24,978 [INFO] HTTP Request: POST http:/

[API retry 1/2000] APITimeoutError: Request timed out.
Sleeping 2.5s before retry...


2026-07-09 12:37:15,135 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 12:37:30,479 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 12:37:35,437 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 12:38:18,422 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 12:38:43,087 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 12:39:14,129 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 12:39:15,885 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 12:39:21,153 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 12:39:41,757 [INFO] HTTP Request: POST http:/

...

[API retry 1/2000] APITimeoutError: Request timed out.
Sleeping 2.5s before retry...


2026-07-09 13:21:57,778 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 13:22:00,020 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 13:22:25,125 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 13:22:25,925 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 13:22:33,488 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 13:22:34,815 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 13:22:44,936 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 13:22:46,426 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 13:22:46,964 [INFO] HTTP Request: POST http:/

[API retry 1/2000] APITimeoutError: Request timed out.
Sleeping 2.5s before retry...


2026-07-09 13:32:28,086 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 13:32:29,388 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 13:32:29,685 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 13:32:33,212 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 13:32:33,851 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 13:32:37,474 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 13:32:40,043 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 13:32:42,750 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 13:32:44,535 [INFO] HTTP Request: POST http:/

{
  "baseline": "sqlstorm",
  "workload_dir": "/mnt/primary/Main/Workloads/sqlstorm_tpcds",
  "num_queries": 100,
  "duration_s": 3840.8
}


## sqlbarber

In [6]:
from workload_generation.baselines import sqlbarber

"""
Generate a workload using the sqlbarber baseline
"""

WORKLOAD_NAME_SQLBARBER = "sqlbarber_{}".format(SCHEMA)

report_sqlbarber = sqlbarber.generate_workload(
    ctx,
    workload_name=WORKLOAD_NAME_SQLBARBER,
    num_queries=NUM_QUERIES,
    warmup=WARMUP,
    random_seed=RANDOM_SEED,
)

print(json.dumps({
    "baseline": report_sqlbarber["baseline"],
    "workload_dir": report_sqlbarber["workload_dir"],
    "num_queries": report_sqlbarber["num_queries"],
    "duration_s": round(report_sqlbarber["duration_s"], 1),
}, indent=2, default=str))


2026-07-09 13:32:53,916 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"


Model warm-up response: ready


2026-07-09 13:33:09,110 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 13:33:10,522 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 13:33:23,783 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 13:33:24,915 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 13:33:56,825 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 13:33:59,334 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 13:34:10,055 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 13:34:11,457 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 13:34:18,540 [INFO] HTTP Request: POST http:/

{
  "baseline": "sqlbarber",
  "workload_dir": "/mnt/primary/Main/Workloads/sqlbarber_tpcds",
  "num_queries": 100,
  "duration_s": 517.2
}


## e2etune

In [7]:
from workload_generation.baselines import e2etune

"""
Generate a workload using the e2etune baseline
"""

WORKLOAD_NAME_E2ETUNE = "e2etune_{}".format(SCHEMA)

report_e2etune = e2etune.generate_workload(
    ctx,
    workload_name=WORKLOAD_NAME_E2ETUNE,
    num_queries=NUM_QUERIES,
    warmup=WARMUP,
    random_seed=RANDOM_SEED,
)

print(json.dumps({
    "baseline": report_e2etune["baseline"],
    "workload_dir": report_e2etune["workload_dir"],
    "num_queries": report_e2etune["num_queries"],
    "duration_s": round(report_e2etune["duration_s"], 1),
}, indent=2, default=str))


2026-07-09 13:41:32,537 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"


Model warm-up response: ready


2026-07-09 13:42:33,157 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 13:42:34,804 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 13:42:38,831 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 13:42:40,422 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 13:42:49,830 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 13:42:58,855 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 13:43:04,722 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 13:43:15,969 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 13:43:16,354 [INFO] HTTP Request: POST http:/

[API retry 1/2000] APITimeoutError: Request timed out.
Sleeping 2.5s before retry...


2026-07-09 13:50:10,756 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 13:50:29,187 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 13:50:36,616 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 13:50:41,934 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 13:51:03,705 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 13:51:20,694 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 13:51:24,310 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 13:51:47,549 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 13:51:48,321 [INFO] HTTP Request: POST http:/

{
  "baseline": "e2etune",
  "workload_dir": "/mnt/primary/Main/Workloads/e2etune_tpcds",
  "num_queries": 100,
  "duration_s": 1456.9
}


## bootstrapping_lcm

In [ ]:
from workload_generation.baselines import bootstrapping_lcm

"""
Generate a workload using the bootstrapping_lcm baseline
"""

WORKLOAD_NAME_BOOTSTRAPPING_LCM = "bootstrapping_lcm_{}".format(SCHEMA)

report_bootstrapping_lcm = bootstrapping_lcm.generate_workload(
    ctx,
    workload_name=WORKLOAD_NAME_BOOTSTRAPPING_LCM,
    num_queries=NUM_QUERIES,
    warmup=WARMUP,
    random_seed=RANDOM_SEED,
)

print(json.dumps({
    "baseline": report_bootstrapping_lcm["baseline"],
    "workload_dir": report_bootstrapping_lcm["workload_dir"],
    "num_queries": report_bootstrapping_lcm["num_queries"],
    "duration_s": round(report_bootstrapping_lcm["duration_s"], 1),
}, indent=2, default=str))


2026-07-09 14:05:51,427 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"


Model warm-up response: ready


2026-07-09 14:07:17,133 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 14:07:30,860 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 14:07:34,481 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 14:07:34,871 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 14:07:37,837 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 14:08:18,101 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 14:08:34,099 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 14:08:48,332 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 14:08:55,386 [INFO] HTTP Request: POST http:/

[API retry 1/2000] APITimeoutError: Request timed out.
Sleeping 2.5s before retry...
[API retry 1/2000] APITimeoutError: Request timed out.
Sleeping 2.5s before retry...


2026-07-09 14:14:25,653 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 14:14:28,646 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 14:14:29,709 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 14:15:07,804 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"


...

2026-07-09 14:27:13,610 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 14:27:25,824 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 14:28:05,177 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 14:28:05,959 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 14:28:17,769 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 14:28:25,523 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 14:28:37,948 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 14:29:14,970 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 14:29:47,586 [INFO] HTTP Request: POST http:/

[API retry 1/2000] APITimeoutError: Request timed out.
Sleeping 2.5s before retry...


2026-07-09 14:32:20,785 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 14:32:27,395 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 14:32:39,204 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 14:32:54,297 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 14:32:59,554 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 14:33:03,778 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 14:33:19,644 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 14:33:33,948 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 14:34:00,681 [INFO] HTTP Request: POST http:/

[API retry 1/2000] APITimeoutError: Request timed out.
Sleeping 2.5s before retry...


2026-07-09 14:36:26,895 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 14:36:39,081 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 14:36:43,600 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 14:36:57,459 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 14:37:24,059 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 14:38:19,963 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 14:38:49,298 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 14:39:02,115 [INFO] HTTP Request: POST http://api.llm.apps.os.dcs.gla.ac.uk/v1/responses "HTTP/1.1 200 OK"
2026-07-09 14:39:07,419 [INFO] HTTP Request: POST http:/

## sql_factory

In [ ]:
from workload_generation.baselines import sql_factory

"""
Generate a workload using the sql_factory baseline
"""

WORKLOAD_NAME_SQL_FACTORY = "sql_factory_{}".format(SCHEMA)

report_sql_factory = sql_factory.generate_workload(
    ctx,
    workload_name=WORKLOAD_NAME_SQL_FACTORY,
    num_queries=NUM_QUERIES,
    warmup=WARMUP,
    random_seed=RANDOM_SEED,
)

print(json.dumps({
    "baseline": report_sql_factory["baseline"],
    "workload_dir": report_sql_factory["workload_dir"],
    "num_queries": report_sql_factory["num_queries"],
    "duration_s": round(report_sql_factory["duration_s"], 1),
}, indent=2, default=str))


# Tear Down Lakehouse

In [2]:
"""
Tear down the lakehouse deployed in the first cell
"""

lh.tear_down()


Successfully tore down lakehouse lakehouse-a
